# The TPU Chip & the Systolic Array — hands-on

A companion notebook for the lesson [*The TPU Chip & the Systolic Array*](https://lms-p-45c03.web.app/topics/ml-systems/tpu-chip-systolic-array/).

You'll **run and measure** the ideas from the lesson instead of just reading them:
data reuse in a systolic array, **bf16 vs fp32**, **arithmetic intensity** (the roofline ridge), and **the padding trap**.

**It runs fully on CPU** — every cell works on the default runtime. The final
benchmark *also* lights up on a real TPU: `Runtime → Change runtime type → v5e-1 TPU`
(a single v5e chip — exactly the single-chip scope of this lesson). The TPU is
availability-gated, so if you can't get one, everything else still runs.

> Tested with JAX 0.4.x. Run the cells top to bottom.

## 1. What hardware did you get?
JAX runs the *same* code on CPU, GPU, or TPU — only the backend changes.

In [ ]:
# Colab ships JAX preinstalled on CPU and TPU runtimes.
# To pin a known-good version, uncomment:  !pip install -q "jax==0.4.30"
import time, math
import numpy as np
import jax, jax.numpy as jnp
import matplotlib.pyplot as plt

print("JAX", jax.__version__)
devices = jax.devices()
PLATFORM = devices[0].platform            # 'cpu', 'gpu', or 'tpu'
HAS_TPU = PLATFORM == "tpu"
print("Devices :", devices)
print("Backend :", PLATFORM.upper(), "| TPU available:", HAS_TPU)
if not HAS_TPU:
    print("\nThis notebook runs fully on CPU. For the real-hardware section,")
    print("select: Runtime -> Change runtime type -> v5e-1 TPU.")

## 2. Why a systolic array? Data reuse

A naive CPU re-reads both operands from memory for *every* multiply-accumulate (MAC) —
that's the *memory wall*. A systolic array streams each input in **once** and reuses it
across the whole grid of processing elements. Count the memory traffic:

In [ ]:
def naive_cpu_reads(N):
    return 2 * N**3            # every MAC fetches A[i,k] and B[k,j] from memory

def systolic_reads(N):
    return 2 * N * N           # each row/column streamed in once, then reused on-chip

for N in (3, 128):
    r1, r2 = naive_cpu_reads(N), systolic_reads(N)
    print(f"N={N:>4}: MACs={N**3:>11,}  naive reads={r1:>13,}  "
          f"systolic reads={r2:>8,}  reuse={r1//r2}x")

print("\nAt N=128 the array reuses each input 128x — exactly the 128x128 MXU.")
print("(Schematic counts: the point is the reuse factor, not a cycle-accurate model.)")

## 3. bf16 vs fp32 — range over precision

The MXU takes **bf16 in** and gives **fp32 out**. bf16 keeps fp32's full **8-bit exponent**
(the same dynamic range) but only 7 mantissa bits. The other 16-bit float, fp16, does the
opposite: more precision, but a 5-bit exponent that **overflows** past ~65504.

In [ ]:
print("value        bf16             fp16")
for v in (1e-7, 1.0, 1e5, 1e8, 1e30):
    bf  = float(jnp.bfloat16(np.float32(v)))
    f16 = float(np.float16(v))
    print(f"{v:>8.0e}    {bf:>12.3e}    {f16:>12.3e}")
print("-> fp16 overflows to inf at 1e5+; bf16 holds, because it kept fp32's exponent.")

**Why the MXU accumulates in fp32.** A bf16 running sum stalls: near 4096, bf16's
spacing between representable numbers is already larger than 1, so adding 1 does nothing.

In [ ]:
acc = jnp.bfloat16(4096.0)
for _ in range(1000):
    acc = acc + jnp.bfloat16(1.0)
print("bf16 accumulate: 4096 + 1x1000 =", float(acc), " (every +1 vanished!)")
print("fp32 accumulate: 4096 + 1x1000 =", 4096.0 + 1000)
print("-> This is why the systolic array multiplies in bf16 but keeps the")
print("   running partial sums in fp32.")

## 4. Arithmetic intensity & the roofline

The array is only worth its FLOPs if you keep it fed. **Arithmetic intensity** =
FLOPs ÷ bytes moved. For an NxN bf16 matmul, FLOPs = 2N³ and bytes ≈ 3N²·2, so
intensity ≈ N/3 — it *grows with N*. Below the hardware's **ridge point** you're
memory-bound (starved); above it you're compute-bound (the good place).

In [ ]:
def intensity(N, dtype_bytes=2):
    return (2 * N**3) / (3 * N * N * dtype_bytes)

RIDGE = 165.0     # TPU v5p: 459 TFLOP/s / 2.8 TB/s
sizes = [128, 256, 512, 1024, 4096, 8192]
print(f"v5p ridge point ~ {RIDGE:.0f} FLOP/byte\n")
for N in sizes:
    ai = intensity(N)
    print(f"N={N:>5}  intensity={ai:8.1f} FLOP/byte  "
          f"{'compute-bound' if ai > RIDGE else 'MEMORY-bound'}")

plt.figure(figsize=(6, 4))
plt.axhline(RIDGE, color="crimson", ls="--", label=f"v5p ridge ~ {RIDGE:.0f}")
plt.plot(sizes, [intensity(N) for N in sizes], "o-", label="matmul intensity")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("matrix size N"); plt.ylabel("FLOP / byte")
plt.title("Small matmuls are memory-bound; big ones cross the ridge")
plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.show()

## 5. The padding trap

XLA zero-pads matmul dimensions up to the MXU tile (**128**). A dimension that's just
over a multiple of 128 silently rounds up — and you pay for the padded FLOPs.

In [ ]:
def padded(n, tile=128):
    return math.ceil(n / tile) * tile

print("dim    padded   wasted FLOPs")
for d in (128, 129, 130, 200, 256, 257, 512):
    p = padded(d)
    waste = (p * p) / (d * d) - 1
    print(f"{d:>4}    {p:>4}      {waste*100:5.1f}%")
print("\n-> A 129x129 matmul pads to 256x256: ~4x the FLOPs for one extra row.")
print("   Choose shapes that are multiples of 128 (and 8 for the vector lanes).")

## 6. A fair CPU-vs-TPU benchmark

The common mistake is to time a function's *first* call — that includes XLA
**compilation**. We warm up **both** devices first, then take the median of several
runs and report **TFLOP/s** (the honest hardware metric).

In [ ]:
def benchmark(N, device, iters=20):
    a = jax.device_put(jax.random.normal(jax.random.PRNGKey(0), (N, N), dtype=jnp.bfloat16), device)
    b = jax.device_put(jax.random.normal(jax.random.PRNGKey(1), (N, N), dtype=jnp.bfloat16), device)
    f = jax.jit(lambda a, b: jnp.dot(a, b, preferred_element_type=jnp.float32))
    f(a, b).block_until_ready()                       # warmup = compile (NOT timed)
    ts = []
    for _ in range(iters):
        t0 = time.perf_counter()
        f(a, b).block_until_ready()
        ts.append(time.perf_counter() - t0)
    med = sorted(ts)[len(ts) // 2]
    return med, (2 * N**3) / med / 1e12               # seconds, TFLOP/s

N = 4096
cpu = jax.devices("cpu")[0]
mc, tc = benchmark(N, cpu)
print(f"CPU : {mc*1e3:7.1f} ms   {tc:6.2f} TFLOP/s")

if HAS_TPU:
    tpu = jax.devices("tpu")[0]
    mt, tt = benchmark(N, tpu)
    print(f"TPU : {mt*1e3:7.1f} ms   {tt:6.2f} TFLOP/s   ({tpu.device_kind})")
    print(f"\n=> The systolic array ran this matmul {mc/mt:.0f}x faster than the CPU.")
else:
    print("TPU : (not connected)")
    print("\nSelect a v5e-1 TPU runtime and re-run this cell to measure real silicon.")

## Put it together

1. **Reuse:** in §2, change `N` to 256 and 512. Why does the reuse factor equal `N`?
2. **Range:** in §3, find the smallest power of ten where **fp16** overflows to `inf` but **bf16** survives. What property of bf16 explains it?
3. **Ridge:** in §4, at roughly which `N` does the matmul cross from memory-bound to compute-bound? Relate that to the v5p ridge of ~165.
4. **Padding:** in §5, you need a 200-wide matmul. What size should you pad *yourself* to, and how much would `200` have wasted?
5. **(TPU)** Grab a `v5e-1` runtime and run §6. How many TFLOP/s do you see, and how does it compare to the CPU?

Back to the lesson → [The TPU Chip & the Systolic Array](https://lms-p-45c03.web.app/topics/ml-systems/tpu-chip-systolic-array/)